### IMPORT

In [1]:
from pathlib import Path

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
import rootutils

import sys
rootutils.setup_root(Path(".").resolve(), indicator=".project-root", pythonpath=True)
sys.path.append(str(Path(".").resolve().parent / "src"))

from bev.models.bev_emb import BEVAdapterViT
from bev.models.bevqa import BEVQA_ViT
from bev.data.bevqa_dataset import BEVQADataset
from bev.models.head import OutputHead
from bev.models.mca import MCALayer
from bev.models.text_emb import TextAdapter
from bev.training.train import train_epoch, val_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"

### PATH

In [2]:
import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf

try:
    initialize(config_path="../configs", version_base=None)
except:
    pass

cfg = compose(config_name="config", overrides=["paths=mini"])

FEAT_DIR = Path(cfg.paths.bev_features_dir)
DICT_DIR = Path(cfg.paths.dict_dir)
GLOVE = Path(cfg.paths.glove_path)

### DATASET

In [3]:
train_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "train_mini",
    json_path=DICT_DIR / "NuScenes_train_questions_mini.json",
    glove=GLOVE
)

In [4]:
val_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "val_mini",
    json_path=DICT_DIR / "NuScenes_val_questions_mini.json",
    glove=GLOVE
)

In [5]:
feat, quest, ans = train_dataset[0]
print(feat.shape, quest.shape, ans)

torch.Size([128, 200, 200]) torch.Size([35, 300]) tensor(26)


### DATALOADER

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [7]:
feat, quest, ans = next(iter(train_dataloader))
print(feat.shape, quest.shape, ans)

torch.Size([4, 128, 200, 200]) torch.Size([4, 35, 300]) tensor([ 8, 17, 29, 18])


### EMBEDDINGS

In [8]:
bev_ad = BEVAdapterViT() # BEV MAP -> BEV EMB
text_ad = TextAdapter() # [B,35,512] to match BEV emb [B,400,512] 
feat_output = bev_ad(feat)
text_output = text_ad(quest)
print(f"Feat:{feat_output.shape}") # [B,100,512]
print(f"Text:{text_output.shape}") # [B,35,512]

Feat:torch.Size([4, 100, 512])
Text:torch.Size([4, 35, 512])


### MODEL

In [9]:
model = BEVQA_ViT()
out = model(feat, quest)
print(out.shape) # [4, 30]

torch.Size([4, 30])
